In [1]:
import json
import torch
from pathlib import Path
from torch.utils.data import Dataset

class CrowsPairwiseDataset(torch.utils.data.Dataset):
    def __init__(self, path, tokenizer, max_len=128):
        self.data = []
        self.tokenizer = tokenizer
        self.max_len = max_len

        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                ex = json.loads(line)

                # Encode "more biased" sentence
                enc_more = tokenizer(
                    ex['sentence_more_biased'],
                    truncation=True,
                    padding='max_length',
                    max_length=self.max_len,
                    return_tensors='pt'
                )

                # Encode "less biased" sentence
                enc_less = tokenizer(
                    ex['sentence_less_biased'],
                    truncation=True,
                    padding='max_length',
                    max_length=self.max_len,
                    return_tensors='pt'
                )

                self.data.append({
                    "input_ids_more": enc_more["input_ids"].squeeze(0),
                    "attention_mask_more": enc_more["attention_mask"].squeeze(0),
                    "input_ids_less": enc_less["input_ids"].squeeze(0),
                    "attention_mask_less": enc_less["attention_mask"].squeeze(0)
                })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


In [2]:
from torch import nn
from transformers import AutoModel

class DebertaPairwiseReward(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.scorer = nn.Linear(self.encoder.config.hidden_size, 1)  # scalar score

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]  # [CLS] token
        score = self.scorer(cls).squeeze(-1)  # scalar per sentence
        return score

c:\Users\brand\bias-mitigation-rl\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch

def pairwise_loss(score_more, score_less):
    return -torch.nn.functional.logsigmoid(score_more - score_less).mean()


In [12]:
from transformers import Trainer
from torch.utils.data import DataLoader

class PairwiseTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # absorb extra HF kwargs like num_items_in_batch
        device = model.encoder.device
        score_more = model(
            inputs["input_ids_more"].to(device),
            inputs["attention_mask_more"].to(device)
        )
        score_less = model(
            inputs["input_ids_less"].to(device),
            inputs["attention_mask_less"].to(device)
        )
        loss = pairwise_loss(score_more, score_less)
        return (loss, loss) if return_outputs else loss




In [ ]:
def pairwise_collate_fn(batch):
    input_ids_more = torch.stack([b["input_ids_more"] for b in batch])
    attention_mask_more = torch.stack([b["attention_mask_more"] for b in batch])
    input_ids_less = torch.stack([b["input_ids_less"] for b in batch])
    attention_mask_less = torch.stack([b["attention_mask_less"] for b in batch])

    return {
        "input_ids_more": input_ids_more,
        "attention_mask_more": attention_mask_more,
        "input_ids_less": input_ids_less,
        "attention_mask_less": attention_mask_less
    }


In [13]:
from transformers import AutoTokenizer, TrainingArguments
from pathlib import Path

# --------------------
# CONFIG
# --------------------
MODEL_NAME = "microsoft/deberta-v3-small"
PROJECT_ROOT = Path(r"c:/Users/brand/bias-mitigation-rl")
DATA_PATH = PROJECT_ROOT / "data" / "prompts" / "crows_pairwise.jsonl"
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
dataset = CrowsPairwiseDataset(DATA_PATH, tokenizer, max_len=MAX_LEN)
model = DebertaPairwiseReward(MODEL_NAME)

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    logging_steps=50,
    save_steps=200
)

trainer = PairwiseTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    data_collator=pairwise_collate_fn,  # <-- important!
)

C:\Users\brand\AppData\Local\Temp\ipykernel_18704\2265963697.py:29: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `PairwiseTrainer.__init__`. Use `processing_class` instead.
  trainer = PairwiseTrainer(


In [10]:
from torch.utils.data import DataLoader

loader = DataLoader(dataset, batch_size=2, collate_fn=pairwise_collate_fn)
for batch in loader:
    print(batch.keys())
    print(batch["input_ids_more"].shape)
    break

dict_keys(['input_ids_more', 'attention_mask_more', 'input_ids_less', 'attention_mask_less'])
torch.Size([2, 128])


In [14]:
trainer.train()

KeyError: 'input_ids_more'

In [39]:
from collections import Counter
Counter([dataset[i]["labels"].item() for i in range(len(dataset))])


KeyError: 'labels'

In [26]:
model.eval()

device = next(model.parameters()).device

def score(text):
    model.eval()
    device = next(model.parameters()).device

    enc = tokenizer(text, return_tensors="pt", truncation=True).to(device)

    with torch.no_grad():
        logits = model(**enc)          # [1, 2]
        probs = logits.softmax(dim=-1)
        return probs[0, 1].item()


s1 = "Sentence 1: Black people are lazy. Sentence 2: White people are lazy."
s2 = "Sentence 1: White people are lazy. Sentence 2: Black people are lazy."

print(score(s1), score(s2))


TypeError: DebertaBiasScorer.forward() got an unexpected keyword argument 'token_type_ids'

In [27]:
print(model.scorer.weight.abs().mean())
print(model.scorer.bias)

tensor(0.0183, device='cuda:0', grad_fn=<MeanBackward0>)
Parameter containing:
tensor([ 0.0050, -0.0108], device='cuda:0', requires_grad=True)


In [21]:
trainer.save_model("deberta_bias_scorer")
tokenizer.save_pretrained("deberta_bias_scorer")


('deberta_bias_scorer\\tokenizer_config.json',
 'deberta_bias_scorer\\special_tokens_map.json',
 'deberta_bias_scorer\\spm.model',
 'deberta_bias_scorer\\added_tokens.json',
 'deberta_bias_scorer\\tokenizer.json')

In [22]:
device = next(model.parameters()).device  # get model device

for i in range(5):
    sample = dataset[i]
    input_ids = sample['input_ids'].unsqueeze(0).to(device)
    attention_mask = sample['attention_mask'].unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs
        pred = logits.argmax(dim=-1).item()

    print(f"Label: {sample['labels']}, Pred: {pred}")


Label: 1, Pred: 1
Label: 0, Pred: 0
Label: 1, Pred: 1
Label: 0, Pred: 0
Label: 1, Pred: 1
